# Week 22: Reproducibility, a search API, Docker, and a model card

Two goals this week: **anyone can reproduce your results from a fresh clone**, and your retrieval system can **run as a service**. That second one is how ML is actually used in companies, and it's a standard interview topic.

| Session | Topic | Time |
|---|---|---|
| Day 1 | Pinned environments and data checksums | 2.5 h |
| Day 2 | A FastAPI search service, with tests | 3 h |
| Day 3 | Docker: package the service | 2 h |
| Day 4 | `reproduce.sh`, a model card, a fresh-clone test; make the repo public | 2 h |
| Review | Spaced recall | 0.5 h |

**Install:** `pip install fastapi uvicorn httpx`. Docker Desktop for Windows (with the WSL 2 backend) is optional: if you can't install it, write the Dockerfile anyway and test it later.

### Done when
- [ ] `requirements-lock.txt` pins exact versions; the datasets are verified by SHA-256.
- [ ] `uvicorn medlab.api:app` serves `/health` and `/search`, and the API tests pass.
- [ ] A `Dockerfile` builds an image that serves the API.
- [ ] `bash scripts/reproduce.sh` regenerates `table_A.md` and `table_B.md` from a fresh clone.
- [ ] `MODEL_CARD.md` exists, and the repo is **public**.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

---
# Day 1: Pin everything (2.5 h)

**Three things must be fixed for a result to be reproducible:** the *code* (a Git commit), the *environment* (exact package versions), and the *data* (exact bytes).
- **Environment:** `pip freeze > requirements-lock.txt` gives exact versions. Keep `pyproject.toml` loose (for people installing your library) and the lock file exact (for reproducing your paper). Record the Python version in the README.
- **Data:** URLs change and files get re-uploaded. Store a **SHA-256 checksum** of every downloaded file and verify it before use.

### Exercise 22.1: `medlab/checksums.py` and `requirements-lock.txt`
```python
sha256_file(path, chunk_size=1 << 20) -> str        # read in chunks: works for files bigger than RAM
verify(path, expected_sha256) -> None               # raise ValueError with both hashes if they differ
```
Compute the hashes of `data/nfcorpus.zip` and the MedMNIST `.npz` file, store them in `DATA_CHECKSUMS.json`, and make the loaders call `verify` after downloading. Add tests, and create the lock file.

In [ ]:
# Implement checksums.py and tests, create requirements-lock.txt, then run the checker.

In [ ]:
import hashlib, tempfile, os
ck = fresh("medlab.checksums")
assert run_tests("tests/test_checksums.py") == 0
with tempfile.NamedTemporaryFile(delete=False) as f:
    f.write(os.urandom(3_000_000)); name = f.name
assert ck.sha256_file(name) == hashlib.sha256(open(name, "rb").read()).hexdigest()
try:
    ck.verify(name, "0" * 64); raise AssertionError("should raise")
except ValueError:
    pass
os.unlink(name)
lock = (CAP / "requirements-lock.txt").read_text()
assert lock.count("==") >= 10
print("✅ 22.1 passed")

---
# Day 2: A search API with FastAPI (3 h)

An **API** lets other programs use your model over HTTP. FastAPI turns type-hinted Python functions into endpoints, and generates interactive docs at `/docs` automatically.

### Exercise 22.2: `capstone/medlab/api.py`
```python
app = FastAPI(title="MedSearch Lab API")
GET /health                 -> {"status": "ok", "n_docs": int}
GET /search?q=...&k=10      -> {"query": str, "results": [{"doc_id", "title", "score"}], "disclaimer": str}
                               k must be 1..100 (use Query(10, ge=1, le=100)); an empty q returns 422
```
Build the index and BM25 **once** at start-up (a module-level object or a cached function), not per request. Test it with `fastapi.testclient.TestClient` in `tests/test_api.py` (CI runs with fake data, so it's fast). Run it for real with:
```bash
uvicorn medlab.api:app --reload
# then open http://127.0.0.1:8000/docs
```

In [ ]:
# Implement api.py and tests, then run the checker.

In [ ]:
assert run_tests("tests/test_api.py") == 0
api = fresh("medlab.api")
from fastapi.testclient import TestClient
client = TestClient(api.app)
assert client.get("/health").json()["status"] == "ok"
r = client.get("/search", params={"q": "cholesterol heart", "k": 5}).json()
assert len(r["results"]) <= 5 and "not clinically validated" in r["disclaimer"].lower()
assert client.get("/search", params={"q": "x", "k": 1000}).status_code == 422
assert client.get("/search", params={"q": ""}).status_code == 422
print("✅ 22.2 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Declare the query parameter as `q: str = Query(..., min_length=1)`. FastAPI then returns 422 for an empty string automatically.

</details>

<details><summary>Hint 2</summary>

Use `@functools.lru_cache(maxsize=1)` on a `get_engine()` function that loads the corpus, builds the index, and returns `(corpus, bm25)`.

</details>


---
# Day 3: Docker (2 h)

A **container** packages your code, its dependencies, and a minimal OS, so it runs the same way everywhere: your laptop, CI, a cloud server. Create `capstone/Dockerfile`:
```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY pyproject.toml README.md ./
COPY medlab ./medlab
RUN pip install --no-cache-dir torch --index-url https://download.pytorch.org/whl/cpu \
 && pip install --no-cache-dir -e . fastapi uvicorn
EXPOSE 8000
CMD ["uvicorn", "medlab.api:app", "--host", "0.0.0.0", "--port", "8000"]
```
and a `.dockerignore` containing `data/`, `.venv/`, `results/raw/`, `.git/`, `__pycache__/`.
```bash
docker build -t medsearch-api .
docker run -p 8000:8000 medsearch-api        # then open http://localhost:8000/docs
```
**Interview question you can now answer:** "What's the difference between a virtual environment and a container?" (A venv isolates Python packages; a container isolates the whole runtime, including the OS libraries.)

In [ ]:
# Write the Dockerfile and .dockerignore (build it if Docker is installed), then run the checker.

In [ ]:
df = (CAP / "Dockerfile").read_text()
assert df.startswith("FROM") and "uvicorn" in df and "EXPOSE 8000" in df
assert "data/" in (CAP / ".dockerignore").read_text()
print("✅ 22.3 passed")

---
# Day 4: One command to reproduce everything, a model card, and going public (2 h)

### Exercise 22.4
1. **`scripts/reproduce.sh`**: `set -euo pipefail`; download and verify the data; run every retrieval config; run every imaging config on val and then test; generate `table_A.md`, `table_B.md` and all figures. Add a comment at the top: expected runtime on CPU, and which step needs a GPU (`resnet_finetune`; provide a `SKIP_GPU=1` switch that reuses the committed results for it).
2. **`MODEL_CARD.md`** for the imaging model (following Mitchell et al., 2019, *Model Cards for Model Reporting*), with the sections: `Model details`, `Intended use`, `Out-of-scope use`, `Training data`, `Evaluation data`, `Metrics`, `Results`, `Ethical considerations`, `Caveats and recommendations`. *Out-of-scope use must include clinical diagnosis.*
3. **The fresh-clone test**: `git clone <your repo> /tmp/fresh && cd /tmp/fresh && python -m venv .venv && source .venv/bin/activate && pip install -r requirements-lock.txt && pip install -e . && SKIP_GPU=1 bash scripts/reproduce.sh`. Compare the tables with the committed ones. Fix *anything* that breaks.
4. Check the repo for secrets and personal files, then make `medsearch-lab` **public**.

In [ ]:
# Create scripts/reproduce.sh and MODEL_CARD.md, then run the checker.

In [ ]:
import os, stat
sh = CAP / "scripts" / "reproduce.sh"
s = sh.read_text()
assert "set -euo pipefail" in s and "medlab.evaluate" in s and "medlab.imaging.run" in s and "SKIP_GPU" in s
assert os.stat(sh).st_mode & stat.S_IXUSR, "make it executable: chmod +x scripts/reproduce.sh"
card = (CAP / "MODEL_CARD.md").read_text()
for h in ["Model details", "Intended use", "Out-of-scope use", "Training data", "Evaluation data", "Metrics", "Results",
          "Ethical considerations", "Caveats and recommendations"]:
    assert h in card, f"missing section {h}"
assert "diagnos" in card.lower()
print("✅ 22.4 passed")

---
# Review (30 min)
1. What three things must be pinned for reproducibility?
2. Why read files in chunks when hashing?
3. Why build the index at start-up rather than per request?
4. Virtual environment vs container: what's the difference?
5. *(Week 12)* What does `if __name__ == "__main__":` do?
6. *(Week 20)* Write a SQL query for the 3 most recent runs of the experiment "imaging".

✍️ *Your answer:* (double-click to edit)

